# 06 — Análisis de errores

Cierra el **criterio de éxito E2 §3**: muestrear los errores de clasificación y conectarlos con los fenómenos lingüísticos identificados en la Entrega 1 (TDT §2). Compara dónde fallan TF-IDF+SVM y DistilBERT sobre el mismo test.

> **No necesita GPU.** Lee las predicciones de BERT desde `reports/preds_test_bert.parquet` (generado por 04); el SVM se recarga desde `models/tfidf_svm.joblib`. Ejecuta 03 y 04 antes.

## 1. Bootstrap

In [ ]:
REPO_URL = 'https://github.com/elvinsomon/pln-poc.git'

import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir('/content/pln-poc'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/pln-poc'], check=True)
    os.chdir('/content/pln-poc')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
import joblib
import pandas as pd
from pathlib import Path
from src.utils.colab import setup_environment, bootstrap_dataset
from src.utils.config import load_config
from src.data.splits import prepare_splits, load_splits

setup_environment(seed=42, project_root=PROJECT_ROOT)
cfg_data = load_config('data.yaml')
cfg = load_config('bert.yaml')
labels = cfg['classes']

# Los splits están persistidos; solo si faltan, regeneramos (requiere el CSV vía Drive).
splits_dir = Path(cfg_data['paths']['splits'])
if not all((splits_dir / f'{n}.parquet').exists() for n in ('train', 'val', 'test')):
    bootstrap_dataset(cfg_data)
    prepare_splits(cfg_data, project_root=PROJECT_ROOT)
print('cwd:', os.getcwd())

## 2. Cargar test + predicciones de ambos modelos

> **Decisión:** BERT se lee del parquet de 04 (no recargamos la GPU); el SVM se reentrena-en-inferencia desde `models/tfidf_svm.joblib`. Si falta el parquet, se recarga el checkpoint de DistilBERT (requiere torch + el modelo en `models/distilbert/`).

In [ ]:
splits = load_splits(cfg_data, project_root=PROJECT_ROOT)
test = splits['test'].copy()

# SVM: barato en CPU.
svm = joblib.load(Path(cfg_data['paths']['models']) / 'tfidf_svm.joblib')
test['pred_svm'] = svm.predict(test['text'])

# BERT: preferimos las predicciones persistidas por 04.
preds_path = Path(cfg['paths']['reports']) / 'preds_test_bert.parquet'
if preds_path.exists():
    test['pred_bert'] = pd.read_parquet(preds_path)['pred_bert'].values
else:
    from src.models.bert import BertClassifier
    bert = BertClassifier.load(Path(cfg['paths']['models']) / 'distilbert')
    test['pred_bert'] = bert.predict(test['text'])

print(test[['text', 'label', 'pred_svm', 'pred_bert']].head())

## 3. Tabla de errores por ejemplo

In [ ]:
test['err_svm']  = test['pred_svm']  != test['label']
test['err_bert'] = test['pred_bert'] != test['label']
test['err_both'] = test['err_svm'] & test['err_bert']
test['err_only_svm']  = test['err_svm']  & ~test['err_bert']   # BERT arregla lo que el SVM falla
test['err_only_bert'] = test['err_bert'] & ~test['err_svm']    # BERT rompe lo que el SVM acierta

print('tasa de error:')
print('  SVM             :', round(test['err_svm'].mean(), 4))
print('  BERT            :', round(test['err_bert'].mean(), 4))
print('  ambos fallan    :', round(test['err_both'].mean(), 4))
print('  solo SVM falla  :', round(test['err_only_svm'].mean(), 4))
print('  solo BERT falla :', round(test['err_only_bert'].mean(), 4))

## 4. Fenómenos lingüísticos (E1 / TDT §2)

> **Decisión:** etiquetamos cada ejemplo con *flags* sobre los placeholders ya presentes en el texto (anonimización C0 + limpieza) y heurísticas ligeras. Permite cruzar errores con fenómenos del E1.

In [ ]:
import re
t = test['text'].str.lower()
test['has_code']  = t.str.contains('<code>',  regex=False)
test['has_url']   = t.str.contains('<url>',   regex=False)
test['has_email'] = t.str.contains('<email>', regex=False)
test['has_user']  = t.str.contains('<user>',  regex=False)
test['short']     = test['text'].str.len() < 60          # títulos cortos / fragmentarios
ES = r'\b(?:que|para|con|por|una|como|pero|gracias|hola|cuando|donde|porque)\b'
test['maybe_codeswitch'] = t.str.contains(ES, regex=True)
test['bug_feature_confusion'] = (
    ((test['label'] == 'bug')     & (test['pred_bert'] == 'feature')) |
    ((test['label'] == 'feature') & (test['pred_bert'] == 'bug'))
)
print(test[['has_code', 'has_url', 'short', 'maybe_codeswitch', 'bug_feature_confusion']].mean().round(4))

## 5. Muestreo de FP/FN por clase (DistilBERT)

In [ ]:
pd.set_option('display.max_colwidth', 90)
for c in labels:
    fn = test[(test['label'] == c) & (test['pred_bert'] != c)]   # falsos negativos de la clase c
    fp = test[(test['label'] != c) & (test['pred_bert'] == c)]   # falsos positivos de la clase c
    print(f'\n===== clase {c}: FN={len(fn)}  FP={len(fp)} =====')
    print('--- FN (era', c, ', BERT dijo otra) ---')
    print(fn.sample(min(3, len(fn)), random_state=cfg_data['seed'])[['text', 'pred_bert', 'pred_svm']].to_string(index=False))
    print('--- FP (no era', c, ', BERT dijo', c, ') ---')
    print(fp.sample(min(3, len(fp)), random_state=cfg_data['seed'])[['text', 'label', 'pred_svm']].to_string(index=False))

## 6. Tasa de error por fenómeno

In [ ]:
flags = ['has_code', 'has_url', 'has_email', 'has_user', 'short', 'maybe_codeswitch']
rows = []
for f in flags:
    sub = test[test[f]]
    if len(sub) == 0:
        continue
    rows.append({'fenomeno': f, 'n': len(sub),
                 'err_svm': round(sub['err_svm'].mean(), 4),
                 'err_bert': round(sub['err_bert'].mean(), 4)})
phen = pd.DataFrame(rows).set_index('fenomeno')
print('tasa de error global  -> SVM:', round(test['err_svm'].mean(), 4),
      '| BERT:', round(test['err_bert'].mean(), 4))
phen

## 7. Persistencia

In [ ]:
cols = ['text', 'label', 'pred_svm', 'pred_bert', 'err_only_svm', 'err_only_bert',
        'has_code', 'has_url', 'short', 'maybe_codeswitch', 'bug_feature_confusion']
errors = test[test['err_svm'] | test['err_bert']][cols]
out = Path(cfg['paths']['reports']) / 'error_analysis_sampled.csv'
out.parent.mkdir(parents=True, exist_ok=True)
errors.sample(min(300, len(errors)), random_state=cfg_data['seed']).to_csv(out, index=False)
print('errores totales:', len(errors), '| muestreados ->', out)

## 8. Lectura lingüística (conexión con E1 / TDT §2)

- **Frontera ilocutiva `feature ↔ question` y `bug ↔ question`** (TDT §2.1, nivel pragmático): vocabulario compartido, intención distinta. Es el grupo de error esperado y donde BERT debería ganar al SVM gracias al contexto.
- **`bug ↔ feature`** (`bug_feature_confusion`): «el botón X no funciona» vs «sería útil un botón X» — misma entidad, polaridad opuesta. Caso testigo de la fuerza ilocutiva del E2.
- **Code-switching ES/EN** (`maybe_codeswitch`): el tokenizer *uncased* en inglés fragmenta el vocabulario español; verificamos si eleva la tasa de error.
- **Ruido de placeholders** (`<CODE>`, `<URL>`, `<EMAIL>`, `<USER>`): el tokenizer los parte en subwords; limitación conocida del preprocesado (no es un bug), señalada para la revisión del E3.
- **Títulos cortos / fragmentarios** (`short`): poco contexto; penaliza más al modelo contextual o al vectorial según la tabla anterior.